In [ ]:
from sklearn import datasets, metrics
import numpy as np
import matplotlib.pyplot as plt

# Chargement du dataset
data = datasets.load_diabetes()

**Chargement des données**

Nous utiliserons le dataset `diabetes` de scikit-learn.

https://scikit-learn.org/stable/datasets/toy_dataset.html#diabetes-dataset

- 10 attributs (features) : age, sex, bmi, bp, s1...s6 (déjà normalisés)
- La cible (target) est une **valeur réelle** (progression de la maladie un an après)
- 442 exemples

In [ ]:
print("Nombre de features :", len(data.feature_names))
print("Noms des features :", data.feature_names)
print("Nombre d'exemples :", data.data.shape[0])
print("Type des targets :", data.target.dtype)

# .data contient uniquement les 10 features (déjà standardisées)
X = data.data
y = data.target

# Vérification
print("\nShape de X :", X.shape)  # (442, 10)
print("Shape de y :", y.shape)   # (442,)

# 5 premières données
print("\n5 premières lignes de X :")
print(X[:5])

# 5 premières étiquettes
print("\n5 premières étiquettes y :")
print(y[:5])

On considère seulement l'attribut **IMC** (index 2).

On met de côté les **50 dernières données** pour tester la qualité d'apprentissage.

In [ ]:
X_train = X[:-50, 2]          # colonne IMC, sauf les 50 dernières
X_test  = X[-50:, 2]          # colonne IMC, 50 dernières
X_train = X_train.reshape(len(X_train), 1)
X_test  = X_test.reshape(len(X_test), 1)
y_train = y[:-50]
y_test  = y[-50:]

In [ ]:
print(X_train[:5])
print(y_train[:5])

**Régression linéaire** : y = ax + b

In [ ]:
from sklearn.linear_model import LinearRegression

# Créer le modèle
model = LinearRegression()

# Entraînement
model.fit(X_train, y_train)

a = model.coef_[0]      # coef de IMC
b = model.intercept_    # ordonnée à l'origine

print("a =", a)
print("b =", b)

**Prédictions et visualisation**

In [ ]:
y_pred_test = model.predict(X_test)

# Représentation graphique
plt.figure(figsize=(10, 6))
plt.plot(X_test, y_test, '.g')
plt.plot(X_test, y_pred_test, '*y')
plt.xlabel("x")
plt.ylabel("y")
plt.grid()
plt.title('Diabète avec données IMC – Régression linéaire')
plt.legend(["Données_test", "Prédiction_test"])
plt.show()

**Calcul MSE et R² sur les données de test**

In [ ]:
from sklearn.metrics import mean_squared_error, r2_score

mse = mean_squared_error(y_test, y_pred_test)
r2  = r2_score(y_test, y_pred_test)
print("MSE sur les données de test :", mse)
print("R²  sur les données de test :", r2)

**Régression par descente de gradient stochastique (SGDRegressor)**

max_iter=10000, tol=1e-3

In [ ]:
from sklearn.linear_model import SGDRegressor

sgd = SGDRegressor(max_iter=10000, tol=1e-3)
sgd.fit(X_train, y_train)

y_pred_sgd = sgd.predict(X_test)

mse_sgd = mean_squared_error(y_test, y_pred_sgd)
r2_sgd  = r2_score(y_test, y_pred_sgd)
print("SGD Regression - MSE:", mse_sgd)
print("SGD Regression - R² :", r2_sgd)

plt.figure(figsize=(10, 6))
plt.plot(X_test, y_test, '.g')
plt.plot(X_test, y_pred_sgd, '*y')
plt.xlabel("x")
plt.ylabel("y")
plt.grid()
plt.title("Régression SGD")
plt.legend(["y_test", "y_pred_sgd"])
plt.show()

**K plus proches voisins** (k = 1, 2, 3, 5)

In [ ]:
from sklearn.neighbors import KNeighborsRegressor

for k in [1, 2, 3, 5]:
    reg = KNeighborsRegressor(n_neighbors=k)
    reg.fit(X_train, y_train)
    y_pred_knn = reg.predict(X_test)
    mse_k = mean_squared_error(y_test, y_pred_knn)
    r2_k  = r2_score(y_test, y_pred_knn)
    print(f"KNN k={k} => MSE : {mse_k:.2f} | R² : {r2_k:.4f}")

# Visualisation pour k=1
reg1 = KNeighborsRegressor(n_neighbors=1)
reg1.fit(X_train, y_train)
y_pred_test = reg1.predict(X_test)

plt.figure(figsize=(10, 6))
plt.plot(X_train, y_train, '+k')
plt.plot(X_test, y_test, '.g')
plt.plot(X_test, y_pred_test, '*r')
plt.xlim(-0.10, 0.2)
plt.xlabel("x")
plt.ylim(0, 350)
plt.ylabel("y")
plt.grid()
plt.title('Diabète avec données IMC k=1')
plt.legend(["Données_train", "Données_test", "Prédiction_test"])
plt.show()

**Observation pour x ∈ [0.1, 0.15]**

- Avec k=1 : la prédiction est la valeur de l'unique voisin le plus proche dans X_train.
- Avec k=2, k=3 : la prédiction est la moyenne des 2 ou 3 voisins les plus proches → la courbe est plus lissée.
- Augmenter k ne garantit pas toujours une meilleure précision : un k trop grand peut sous-apprendre (biais élevé).

**SVR (Support Vector Regression)**

In [ ]:
from sklearn.svm import SVR

svr = SVR()
svr.fit(X_train, y_train)
y_pred_svr = svr.predict(X_test)

mse_svr = mean_squared_error(y_test, y_pred_svr)
r2_svr  = r2_score(y_test, y_pred_svr)
print("SVR - MSE:", mse_svr)
print("SVR - R² :", r2_svr)

x_range = np.linspace(X_train.min(), X_train.max(), 300).reshape(-1, 1)
plt.figure(figsize=(10, 6))
plt.scatter(X_test, y_test, c='green', s=30, label='Données test')
plt.plot(x_range, svr.predict(x_range), color='blue', linewidth=2, label='SVR')
plt.xlabel("x")
plt.ylabel("y")
plt.grid()
plt.title('Diabète avec données IMC – SVR')
plt.legend()
plt.show()

**Prétraitement pour la méthode au pire score**

Note : les données du dataset diabetes sont **déjà standardisées**, donc ce prétraitement est redondant ici mais il est instructif de le vérifier.

In [ ]:
from sklearn.preprocessing import StandardScaler

scaler = StandardScaler()
X_train_s = scaler.fit_transform(X_train)
X_test_s  = scaler.transform(X_test)

print("Moyenne (train standardisé) :", X_train_s.mean().round(6))   # ≈ 0
print("Écart-type (train standardisé) :", X_train_s.std().round(6)) # ≈ 1

# Ré-entraîner SVR sur données standardisées
svr_s = SVR()
svr_s.fit(X_train_s, y_train)
y_pred_svr_s = svr_s.predict(X_test_s)

mse_svr_s = mean_squared_error(y_test, y_pred_svr_s)
r2_svr_s  = r2_score(y_test, y_pred_svr_s)
print("SVR (données standardisées) - MSE:", mse_svr_s)
print("SVR (données standardisées) - R² :", r2_svr_s)

**Remarque** : les données du dataset sont déjà prétraitées. On peut le vérifier via `data.DESCR`.

In [ ]:
print(data.DESCR)

**Apprentissage avec TOUS les attributs**

On change X_train et X_test pour utiliser toutes les colonnes.

In [ ]:
# Nouvelle partition avec tous les attributs
X_train_all = X[:-50]   # shape (392, 10)
X_test_all  = X[-50:]   # shape (50, 10)
# y_train et y_test restent identiques

results = {}

# --- Régression linéaire ---
lr_all = LinearRegression()
lr_all.fit(X_train_all, y_train)
y_pred_lr = lr_all.predict(X_test_all)
results['LinearRegression'] = {
    'MSE': mean_squared_error(y_test, y_pred_lr),
    'R2':  r2_score(y_test, y_pred_lr)
}

# --- SVR ---
svr_all = SVR()
svr_all.fit(X_train_all, y_train)
y_pred_svr_all = svr_all.predict(X_test_all)
results['SVR'] = {
    'MSE': mean_squared_error(y_test, y_pred_svr_all),
    'R2':  r2_score(y_test, y_pred_svr_all)
}

# --- KNN k=1,2,3,4 ---
for k in [1, 2, 3, 4]:
    knn_all = KNeighborsRegressor(n_neighbors=k)
    knn_all.fit(X_train_all, y_train)
    y_pred_knn_all = knn_all.predict(X_test_all)
    results[f'KNN k={k}'] = {
        'MSE': mean_squared_error(y_test, y_pred_knn_all),
        'R2':  r2_score(y_test, y_pred_knn_all)
    }

# --- Affichage ---
print(f"{'Méthode':<20} {'MSE':>12} {'R²':>10}")
print("-" * 44)
for method, scores in results.items():
    print(f"{method:<20} {scores['MSE']:>12.2f} {scores['R2']:>10.4f}")